In [1]:
import os
import sys
import requests
import datetime as dt
import numpy as np
from dotenv import load_dotenv
from pathlib import Path
import requests
import pandas as pd
import talib as ta
import plotly.graph_objs as go

import ipywidgets as widgets
from ipywidgets import Dropdown, Text, Button, Output
from IPython.display import display

from Modules.utility import Utility
from Modules.show_plot import ShowPlot
from Modules.reques_api import RequestApi
from Modules.get_market_data import GetMarketData
from Modules.stock_prices_and_market_data import ClassStockPricesMarketData
from Modules.financial import Financial
from Modules.pdf_url_to_markdown import PDFUrlToMarkdown
from Modules.webpage_to_markdown import WebpageToMarkdown

In [2]:
API_BASE_URL = os.getenv("API_BASE_URL", "http://localhost:8000")
request_api = RequestApi(API_BASE_URL)
get_market_data = GetMarketData(Path('/workspace/data'))
utility = Utility()
stock_prices_market_data = ClassStockPricesMarketData()
financial = Financial()
# URLからPDFをダウンロードしてMarkdownに変換するクラスのインスタンスを作成
pdf_to_md = PDFUrlToMarkdown()
# WEBページをMarkdownに変換する関数
webpage_to_markdown = WebpageToMarkdown()

In [5]:
dsv = 'DSV'
market = 'TO'
start = '1999-01-01'
end = dt.datetime.now().strftime('%Y-%m-%d')

response = request_api.update_stock_timeseries_data(
    code=dsv,
    market=market,
    start=start,
    end=end
)
response

{'result': True}

In [6]:
dsv_timeseries_df = request_api.get_stock_time_series_data(
    code=dsv,
    market=market,
    start=start,
    end=end
)
dsv_timeseries_df

取得件数: 2891


,id,stock_code,stock_market,date,open,high,low,close,volume,ma5,...,upper1,lower1,cross,gc,dc,macd_gc,macd_dc,rci_gc,rci_dc,rising_condition
0,1112895,DSV,TO,2014-10-24,0.06,0.06,0.06,0.06,1500,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
1,1112896,DSV,TO,2014-10-27,0.06,0.06,0.06,0.06,0,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
2,1112897,DSV,TO,2014-10-28,0.06,0.06,0.06,0.06,0,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
3,1112898,DSV,TO,2014-10-29,0.06,0.06,0.06,0.06,0,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
4,1112899,DSV,TO,2014-10-30,0.06,0.06,0.06,0.06,0,0.060,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2886,1167003,DSV,TO,2026-04-27,9.21,9.40,9.06,9.40,1327800,9.870,...,10.317441,8.416159,True,NaN,NaN,NaN,0.231111,NaN,NaN,False
2887,1167004,DSV,TO,2026-04-28,8.71,9.11,8.55,9.07,3774800,9.602,...,10.287152,8.595248,True,NaN,NaN,NaN,NaN,NaN,NaN,False
2888,1167005,DSV,TO,2026-04-29,8.31,8.54,8.28,8.50,3331900,9.288,...,10.268081,8.675119,False,NaN,9.4716,NaN,NaN,NaN,NaN,False
2889,1167006,DSV,TO,2026-04-30,8.38,8.83,8.25,8.70,6498500,9.030,...,10.267655,8.687545,False,NaN,NaN,NaN,NaN,NaN,NaN,False


In [7]:
def stock_prices_and_gold_prices(
        code: str,
        name: str,
        start: str,
        end: str,
        df_sp: pd.DataFrame | None = None,
        df_gold: pd.DataFrame | None = None,
        df_silver: pd.DataFrame | None = None
    ):
    # /api/v1/time_series_data/stock/
    response = request_api.get_stock_time_series_data(
        code=code,
        market=None,
        start=start,
        end=end
    )
    df_stock = pd.DataFrame(response)

    # S&P500を統合
    if df_sp is not None:
        df_sp_tmp = df_sp.copy() if df_sp is not None else pd.DataFrame()
        if "date" not in df_sp_tmp.columns:
            df_sp_tmp = df_sp_tmp.reset_index()

        df_sp_tmp["date"] = pd.to_datetime(df_sp_tmp["date"])
        df_sp_tmp = df_sp_tmp.set_index("date")
        df_sp_tmp = df_sp_tmp.loc[start:end]

    # 金価格を統合
    if df_gold is not None:
        df_gold_tmp = df_gold.copy() if df_gold is not None else pd.DataFrame()
        if "date" not in df_gold_tmp.columns:
            df_gold_tmp = df_gold_tmp.reset_index()
        df_gold_tmp["date"] = pd.to_datetime(df_gold_tmp["date"])
        df_gold_tmp = df_gold_tmp.set_index("date")
        df_gold_tmp = df_gold_tmp.loc[start:end]

    # 銀価格を統合
    if df_silver is not None:
        df_silver_tmp = df_silver.copy() if df_silver is not None else pd.DataFrame()
        if "date" not in df_silver_tmp.columns:
            df_silver_tmp = df_silver_tmp.reset_index()
        df_silver_tmp["date"] = pd.to_datetime(df_silver_tmp["date"])
        df_silver_tmp = df_silver_tmp.set_index("date")
        df_silver_tmp = df_silver_tmp.loc[start:end]

    # 金価格
    df = df_stock.copy()
    df["date"] = pd.to_datetime(df["date"])
    df = df.set_index("date")
    df = df.loc[start:end]


    # インデックスを揃えて結合
    if df_sp is not None:
        df["MA5_SP"] = df_sp_tmp["ma5"].reindex(df.index)
        df["MA25_SP"] = df_sp_tmp["ma25"].reindex(df.index)
    if df_gold is not None:
        df["MA5_GOLD"] = df_gold_tmp["ma5"].reindex(df.index)
        df["MA25_GOLD"] = df_gold_tmp["ma25"].reindex(df.index)
    if df_silver is not None:
        df["MA5_SILVER"] = df_silver_tmp["ma5"].reindex(df.index)
        df["MA25_SILVER"] = df_silver_tmp["ma25"].reindex(df.index)

    show_plot = ShowPlot()
    fig = show_plot.create_basic_chart(
        df=df.reset_index(),
        code=code,
        name=name,
        start=start,
        end=end
    )
    # ★ 2つのY軸を定義（左：HYMC、右：SP500 & GOLD）
    fig.update_layout(
        yaxis=dict(
            title=f"{name} Price",
            side="left"
        ),
        yaxis2=dict(
            title="SP500 / GOLD",
            overlaying="y",
            side="right"
        )
    )

    # --- SP500（右軸） ---
    if df_sp is not None:
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA5_SP"],
                name="SP_MA5",
                line={"color": "blue", "width": 1.2},
                yaxis="y2"   # ★ 右軸
            )
        )
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA25_SP"],
                name="SP_MA25",
                line={"color": "gray", "width": 1.2},
                yaxis="y2"   # ★ 右軸
            )
        )

    # --- GOLD（右軸） ---
    if df_gold is not None:
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA5_GOLD"],
                name="GOLD_MA5",
                line={"color": "orange", "width": 1.2},
                yaxis="y2"   # ★ 右軸
            )
        )
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA25_GOLD"],
                name="GOLD_MA25",
                line={"color": "yellow", "width": 1.2},
                yaxis="y2"   # ★ 右軸
            )
        )

    # --- SILVER（左軸） ---
    if df_silver is not None:
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA5_SILVER"],
                name="SILVER_MA5",
                line={"color": "gray", "width": 1.2},
                yaxis="y"   # ★ 左軸
            )
        )
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA25_SILVER"],
                name="SILVER_MA25",
                line={"color": "black", "width": 1.2},
                yaxis="y"   # ★ 左軸
            )
        )

    return fig

In [8]:
name = "Silver Mountain Resources Inc"
start = dt.datetime(2025, 1, 1).strftime("%Y-%m-%d")
end = dt.datetime(2026, 4, 17).strftime("%Y-%m-%d")
# グラフ領域の作成
fig = stock_prices_and_gold_prices(
    code=dsv,
    name=name,
    start=start,
    end=end,
    df_sp=None,
    df_gold=None,
    df_silver=None
)
fig.show()

取得件数: 531


In [9]:
response = request_api.update_corp_finance_data(
    code=dsv,
    market=market
)
response

{'result': True}

In [10]:
dsv_financials_data = request_api.get_corp_financials_data(code=dsv, market=market)
dsv_balance_sheet_data = request_api.get_corp_balance_sheet_data(code=dsv, market=market)
dsv_cash_flow_data = request_api.get_corp_cash_flow_data(code=dsv, market=market)
dsv_earnings_data = request_api.get_corp_earnings_data(code=dsv, market=market)
dsv_quarterly_earnings_data = request_api.get_corp_quarterly_earnings_data(code=dsv, market=market)

In [11]:
# ４年分の財務データ
dsv_financials_data_df = pd.DataFrame(dsv_financials_data['results'])
# ４年分のバランスシート
dsv_balance_sheet_data_df = pd.DataFrame(dsv_balance_sheet_data['results'])
# ４年分のキャッシュフロー
dsv_cash_flow_data_df = pd.DataFrame(dsv_cash_flow_data['results'])
# ４年分の収益データ
dsv_earnings_data_df = pd.DataFrame(dsv_earnings_data['results'])
# ４年分の四半期収益データ
dsv_quarterly_earnings_data_df = pd.DataFrame(dsv_quarterly_earnings_data['results'])

In [12]:
"""
◆ 1. 株価・市場データ
• 現在株価（Price）
• 時価総額（Market Cap）
• 出来高（Volume）
• 52週高値・安値
• Beta（ボラティリティ指標）ß
"""
stock_prices_market_data.stock_prices_and_market_data(
    code=dsv,
    market=market,
    bs_df=dsv_balance_sheet_data_df
)

取得件数: 458
取得件数: 461
取得件数: 457
取得件数: 460
取得件数: 457
取得件数: 458
取得件数: 458
取得件数: 459
取得件数: 458
取得件数: 458


,close,market_cap,shares_outstanding,higher_rate_par_52_weeks,lower_rate_par_52_weeks,beta
0,0.51,NaN,NaN,2.84,0.22,1.781374
1,2.00,7.038832e+08,351941580.0,2.70,0.87,0.854891
2,1.98,7.838073e+08,395862249.0,2.16,0.56,0.826258
3,1.09,4.365028e+08,400461244.0,1.38,0.52,0.577737
4,0.70,5.654320e+08,807760000.0,9.54,0.52,0.750320


In [13]:
"""
◆ 2. 財務データ（Financials）+ EPS（Earnings Per Share）+ PBR（Price-to-Book Ratio）
• 売上高（Revenue）
• 営業利益（Operating Income）
• 純利益（Net Income）
• EBITDA（企業による）
• 総資産（Total Assets）
• 総負債（Total Liabilities）
• 現金（Cash）
• 希釈EPS（Diluted EPS）
• 基本EPS（Basic EPS）
• 営業キャッシュフロー（Operating Cash Flow）
• フリーキャッシュフロー（Free Cash Flow）
"""
financial_df = financial.calc_financial(
    code = dsv,
    market = market,
)
financial_df

取得件数: 2042


,date,revenue,earnings,total_assets,total_debt,cash_and_cash_equivalents,EBITDA,operating_income,basic_eps,diluted_eps,operating_cash_flow,free_cash_flow
0,2021-12-31,NaN,NaN,NaN,NaN,NaN,1.503859e+05,NaN,NaN,NaN,NaN,NaN
1,2022-12-31,0.0,-3.027982e+07,6.747961e+07,4.050980e+05,3.405610e+07,-3.006438e+07,-3.394758e+07,-0.088417,-0.088417,-2.881366e+07,-2.911470e+07
2,2023-12-31,0.0,-1.187435e+07,1.104390e+08,8.400000e+04,4.456700e+07,-1.158581e+07,-1.478775e+07,-0.030152,-0.030152,-1.149973e+06,-2.666499e+07
3,2024-12-31,0.0,-1.516700e+07,8.540100e+07,1.018000e+06,2.037000e+07,-1.382300e+07,-1.410000e+07,-0.040000,-0.040000,-1.514100e+07,-2.239100e+07
4,2025-12-31,653213000.0,1.068100e+08,1.795851e+09,6.594000e+06,4.106670e+08,2.901470e+08,1.913230e+08,0.160000,0.150000,3.777230e+08,1.721910e+08


In [14]:
# PBR（Price-to-Book Ratio）やROE（Return on Equity）などの投資指標を計算
financial.calc_stock_investment_indicators(code=agmr, market=market)

取得件数: 457


,date,EV,reason,BPS,PBR,ROE,operating_income,basic_eps,diluted_eps
0,2021-12-31,NaN,no_price,NaN,NaN,NaN,NaN,NaN,NaN
1,2022-12-31,NaN,no_price,NaN,NaN,-0.460932,-3.394758e+07,-0.088417,-0.088417
2,2023-12-31,NaN,no_price,NaN,NaN,-0.121349,-1.478775e+07,-0.030152,-0.030152
3,2024-12-31,NaN,NaN,0.194346,3.859099,-0.194878,-1.410000e+07,-0.040000,-0.040000
4,2025-12-31,NaN,NaN,0.776822,10.851903,0.170219,1.913230e+08,0.160000,0.150000


In [16]:
md_file_path =pdf_to_md.pdf_url_to_markdown(
    pdf_url="https://discoverysilver.com/site/assets/files/6684/2025-q4-dsv-fs.pdf",
    directory_path="/workspace/data",
)
md_file_path

Generated: /workspace/data/2025-q4-dsv-fs.pdf.md


'/workspace/data/2025-q4-dsv-fs.pdf.md'

In [18]:
md_file_path =pdf_to_md.pdf_url_to_markdown(
    pdf_url="https://discoverysilver.com/site/assets/files/6684/2025-q4-dsv-mda.pdf",
    directory_path="/workspace/data",
)
md_file_path

Generated: /workspace/data/2025-q4-dsv-mda.pdf.md


'/workspace/data/2025-q4-dsv-mda.pdf.md'

In [19]:
md_file_path =pdf_to_md.pdf_url_to_markdown(
    pdf_url="https://discoverysilver.com/site/assets/files/6684/discovery-annual-information-form-february-19-2026-for-website.pdf",
    directory_path="/workspace/data",
)
md_file_path

markitdown failed (rc=-9). Falling back to pdftotext if available. stderr:
onnxruntime cpuid_info warning: Unknown CPU vendor. cpuinfo_vendor value: 0
Generated (pdftotext fallback): /workspace/data/discovery-annual-information-form-february-19-2026-for-website.pdf.md


'/workspace/data/discovery-annual-information-form-february-19-2026-for-website.pdf.md'

In [20]:
md_file_path =pdf_to_md.pdf_url_to_markdown(
    pdf_url="https://discoverysilver.com/site/assets/files/6239/cordero_silver_project_ni_43-101_technical_report_final.pdf",
    directory_path="/workspace/data",
)
md_file_path

markitdown failed (rc=-9). Falling back to pdftotext if available. stderr:
onnxruntime cpuid_info warning: Unknown CPU vendor. cpuinfo_vendor value: 0
Generated (pdftotext fallback): /workspace/data/cordero_silver_project_ni_43-101_technical_report_final.pdf.md


'/workspace/data/cordero_silver_project_ni_43-101_technical_report_final.pdf.md'